In [ ]:
import os
import base64
from pathlib import Path
import math

from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from IPython.display import display, Image
from pydantic import BaseModel, Field
from PIL import Image as PIL_IMAGE

load_dotenv(find_dotenv())

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
MODEL_NAME = "gpt-5-mini"

In [ ]:
def encode_image(image_path: str) -> bytes:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def stitch_images_from_folder(
    folder_path,
    output_path="stitched_output.jpg",
    layout="grid",
    max_images=10,
    padding=10,
    background_color=(255, 255, 255),
):
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder}")

    if not folder.is_dir():
        raise NotADirectoryError(f"Path is not a folder: {folder}")

    image_paths = sorted(
        [
            path
            for path in folder.iterdir()
            if path.suffix.lower() in SUPPORTED_EXTENSIONS
        ]
    )

    if not image_paths:
        raise ValueError("No supported images found in the folder.")

    if len(image_paths) > max_images:
        print(f"Found {len(image_paths)} images. Using first {max_images}.")
        image_paths = image_paths[:max_images]

    images = []

    for path in image_paths:
        img = PIL_IMAGE.open(path).convert("RGB")
        images.append(img)

    # Resize all images to the same size based on the smallest image
    min_width = min(img.width for img in images)
    min_height = min(img.height for img in images)

    resized_images = [
        img.resize((min_width, min_height), PIL_IMAGE.LANCZOS) for img in images
    ]

    count = len(resized_images)

    if layout == "horizontal":
        cols = count
        rows = 1

    elif layout == "vertical":
        cols = 1
        rows = count

    elif layout == "grid":
        cols = math.ceil(math.sqrt(count))
        rows = math.ceil(count / cols)

    else:
        raise ValueError("layout must be 'grid', 'horizontal', or 'vertical'.")

    final_width = cols * min_width + padding * (cols + 1)
    final_height = rows * min_height + padding * (rows + 1)

    stitched_image = PIL_IMAGE.new("RGB", (final_width, final_height), background_color)

    for index, img in enumerate(resized_images):
        row = index // cols
        col = index % cols

        x = padding + col * (min_width + padding)
        y = padding + row * (min_height + padding)

        stitched_image.paste(img, (x, y))

    stitched_image.save(output_path)

    print(f"Used {len(image_paths)} images:")
    for path in image_paths:
        print(f"- {path.name}")

    print(f"\nSaved stitched image to: {output_path}")

In [ ]:
stitch_images_from_folder(
    folder_path="inputs",
    output_path="output/brand_image.jpg",
    layout="grid",
    padding=20,
)

# Tool 1: brand_analysis_tool

In [ ]:
class BrandAnalysis(BaseModel):
    colors: str = Field(description="Specific color tokens with hex values where visible or safely inferred.")
    typography: str = Field(description="Observed type style, weights, scale, hierarchy, and any uncertainty.")
    icon_style: str = Field(description="Logo, icon, illustration, and visual asset style.")
    layout_patterns: str = Field(description="Observed page structure, grid, density, section rhythm, and responsive assumptions.")
    ui_elements: str = Field(description="Buttons, cards, forms, nav, borders, shadows, spacing, and interaction patterns.")
    content_patterns: str = Field(description="Visible brand name, section names, CTA wording, domain language, and repeated content types.")
    brand_voice: str = Field(description="5-7 concise personality traits grounded in the image.")
    must_preserve: str = Field(description="Concrete visual or content details the generated UI must keep.")
    avoid: str = Field(description="Details that would break brand fit or look generic.")

In [ ]:
def brand_analysis_tool(brand_guidelines_image: str) -> BrandAnalysis:

    image_analysis_prompt = """
    You are a senior brand systems designer. Analyze the attached brand reference
    image and extract only design evidence that is useful for generating a new
    high-fidelity web UI mockup.

    The image may be a stitched board containing multiple screenshots. Treat it
    as a visual reference, not as a single page to copy.

    Extraction rules:
    - Separate observed facts from reasonable inferences.
    - Use exact text, section names, CTA labels, product/person names, and domain
      language when they are visible.
    - Use hex values when colors are visible enough to estimate; otherwise give
      precise color descriptions and mark uncertainty.
    - Do not invent font families, brand names, logos, or content. Say
      "unknown" when the image does not support a detail.
    - Capture repeatable design tokens and patterns, not one-off screenshot
      artifacts such as browser chrome.
    - Favor concrete constraints that a later image-generation prompt can reuse.

    Return a compact but complete brand system covering colors, typography,
    icon/logo style, layout patterns, UI components, content patterns, brand
    voice, must-preserve details, and things to avoid.
    """

    base64_image = encode_image(brand_guidelines_image)

    response = client.responses.parse(
        model=MODEL_NAME,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": image_analysis_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            }
        ],
        text_format=BrandAnalysis,
    )
    return response.output_parsed

In [ ]:
brand_guidelines_image = "output/brand_image.jpg"
display(Image(brand_guidelines_image, width=400))

In [ ]:
brand_analysis = brand_analysis_tool(brand_guidelines_image)
brand_analysis

# Tool 2: generate_design_concepts

In [ ]:
from typing import List


class ConceptInfo(BaseModel):
    title: str
    differentiation_angle: str
    description: str
    prompt: str
    acceptance_criteria: List[str]
    negative_prompt: str


class Ideas(BaseModel):
    ideas: List[ConceptInfo]

In [ ]:
def generate_design_concepts(
    user_description: str,
    brand_guidelines_image: str,
    brand_analysis: BrandAnalysis,
    num_concepts: int = 3,
) -> Ideas:

    design_concept_prompt = f"""
    You are a senior product designer creating multiple exploratory UI
    directions from one brand system.

    User request:
    {user_description}

    Brand analysis JSON:
    {brand_analysis.model_dump_json()}

    Generate exactly {num_concepts} distinct web UI design concepts. Each
    concept must use the same brand guidelines, but the layout architecture,
    content emphasis, hero composition, section order, and interaction model
    must be meaningfully different.

    Shared brand requirements for every concept:
    - Preserve the brand's visible identity, colors, type hierarchy, spacing,
      section rhythm, CTA style, card style, icon style, and content language.
    - Reuse visible names, labels, navigation items, services, project/tutorial
      categories, and CTA wording when present in the brand analysis.
    - Treat the reference as a brand guideline, not as a layout template to
      duplicate.
    - Do not use lorem ipsum, placeholder names, generic metrics, fake stock
      content, or unrelated decorative imagery.
    - If the user request lacks details, fill gaps with brand-consistent content
      from the reference instead of generic filler.
    - Specify the intended viewport as a landscape desktop web UI mockup.
    - Do not ask for browser chrome, OS chrome, watermarks, captions, or a
      moodboard collage unless the user explicitly requested them.
    - Include layout details, visual hierarchy, component details, spacing,
      color tokens, typography, and accessibility/contrast expectations.

    Differentiation requirements:
    - Do not create minor variants of the same landing page.
    - Do not simply recreate the reference screenshots with cleaner copy.
    - Each concept needs a different strategic angle, such as a conversion-led
      consultation page, an editorial/storytelling portfolio, a technical
      systems dashboard, a case-study-first proof page, or a course/media-led
      learning hub.
    - Keep brand tokens stable; vary composition, hierarchy, density, content
      sequencing, and primary user journey.

    Return:
    - title: short concept name.
    - differentiation_angle: one sentence explaining how this concept differs
      from the other concepts and from the reference layout.
    - description: product/design rationale in plain language.
    - prompt: a direct image-generation prompt for the mockup.
    - acceptance_criteria: 5-8 concrete checks the mockup should pass.
    - negative_prompt: concise list of visual/content failures to avoid.
    """

    base64_image = encode_image(brand_guidelines_image)

    response = client.responses.parse(
        model=MODEL_NAME,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": design_concept_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            }
        ],
        text_format=Ideas,
    )

    return response.output_parsed

In [ ]:
user_description = """Create landing page for personal portfolio web ui."""

In [ ]:
ideas = generate_design_concepts(
    user_description, brand_guidelines_image, brand_analysis
)

In [ ]:
[(idea.title, idea.differentiation_angle) for idea in ideas.ideas]

# Tool 3: generate_idea_image

In [ ]:
def generate_idea_image(
    concept: str,
    brand_guidelines_image: str,
    output_path: str = "output/idea_image.png",
) -> str | None:

    concept_image_prompt = f"""
    Generate one high-fidelity landscape desktop web UI mockup from the design
    brief below.

    Attached image: brand/style reference only. Use it to preserve visual DNA,
    tokens, density, content patterns, and component style. The design brief is
    the authority for layout and concept direction. Do not copy the reference
    page structure, browser chrome, or create a collage of screenshots.

    Design brief:
    {concept}

    Rendering requirements:
    - Output should look like a polished product screenshot of the new web UI
      concept itself.
    - Use brand-specific names, labels, CTAs, services, project/tutorial content,
      and metrics from the brief/reference when available.
    - No lorem ipsum, no placeholder [Name], no generic fake portfolio content,
      no unrelated stock portrait, no watermark, no browser chrome.
    - Keep text short, realistic, and legible. Favor fewer readable labels over
      many tiny unreadable blocks.
    - Match the brand color palette, typography hierarchy, spacing rhythm,
      border radius, card treatment, icon style, and contrast level.
    - Make the UI direction visibly distinct from the reference screenshots:
      use the concept's own layout, section order, hero composition, and user
      journey while retaining the same brand system.
    - Show the primary first viewport and a clear hint of the next section.
    """
    base64_image = encode_image(brand_guidelines_image)

    response = client.responses.create(
        model=MODEL_NAME,
        input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": concept_image_prompt },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
        tools=[{"type": "image_generation", "model": "gpt-image-1-mini", "action": "generate", "size": "1536x1024"}],
    )

    # Save the image to a file
    image_data = [
        output.result
        for output in response.output
        if output.type == "image_generation_call"
    ]

    if image_data:
        image_base64 = image_data[0]
        image_path = Path(output_path)
        image_path.parent.mkdir(parents=True, exist_ok=True)
        with open(image_path, "wb") as f:
            f.write(base64.b64decode(image_base64))
            
        return str(image_path)
    
    return None
        

In [ ]:
idea_prompt = f"""{ideas.ideas[0].prompt}

Negative instructions:
{ideas.ideas[0].negative_prompt}
"""

idea_image_path = generate_idea_image(
    idea_prompt,
    brand_guidelines_image
)

In [ ]:
display(Image(filename=idea_image_path, width=400))

# Tool 4: `evaluate_image`

In [ ]:
class EvaluationResult(BaseModel):
    brand_fidelity: float
    visual_aesthetic: float
    contrast: float
    repetition: float
    alignment: float
    proximity: float
    content_specificity: float
    concept_distinctiveness: float
    overall: float
    is_passing: bool
    feedback: str
    prompt_improvements: str

In [ ]:
def evaluate_image(
    threshold: float, brand_guidelines_image: str, idea_image: str
) -> EvaluationResult:

    evaluation_prompt = f"""
    You are a strict senior design QA reviewer.

    Inputs:
    - Image 1 is the brand reference.
    - Image 2 is the generated UI mockup to evaluate.

    Evaluate whether Image 2 could belong to the same brand system as Image 1
    while still presenting a meaningfully new UI idea. Be critical: a visually
    pleasant image should fail if it uses generic placeholder content, ignores
    the brand identity, changes the component language, has weak contrast,
    copies the reference layout too closely, or cannot be used as a retry
    signal.

    Score each criterion from 1.0 (poor) to 5.0 (excellent):
    - brand_fidelity: colors, logo/mark treatment, typography feel, content
      language, and recognizable brand details match the reference.
    - visual_aesthetic: overall polish, hierarchy, balance, and professional UI
      quality.
    - contrast: text and controls are readable; important elements stand apart.
    - repetition: cards, buttons, icons, headings, radii, borders, and spacing
      repeat consistently.
    - alignment: grid, section edges, baselines, and horizontal/vertical flow are
      coherent.
    - proximity: related elements are grouped logically without awkward gaps.
    - content_specificity: copy, labels, metrics, media, and sections feel
      specific to the brand/user request, not generic filler.
    - concept_distinctiveness: the mockup explores a new layout, section
      sequence, hero composition, or user journey instead of near-copying the
      reference screenshots.

    Overall is the average of the eight scores. is_passing must be true only if
    overall >= {threshold} and no individual criterion is below 4.0.

    Feedback rules:
    - If passing, feedback should be "Looks good" and prompt_improvements should
      be empty.
    - If failing, feedback should explain the main visual/content problems.
    - prompt_improvements should be directly reusable as instructions for the
      next image-generation attempt. Focus on concrete changes, not general
      design advice.
    - If concept_distinctiveness is low, explain how to preserve the brand
      while changing composition, hierarchy, or journey.
    """

    brand_guidelines_image_b64 = encode_image(brand_guidelines_image)
    idea_image_b64 = encode_image(idea_image)

    response = client.responses.parse(
        model=MODEL_NAME,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": evaluation_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{brand_guidelines_image_b64}",
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{idea_image_b64}",
                    },
                ],
            }
        ],
        text_format=EvaluationResult
    )

    result = response.output_parsed
    scores = [
        result.brand_fidelity,
        result.visual_aesthetic,
        result.contrast,
        result.repetition,
        result.alignment,
        result.proximity,
        result.content_specificity,
        result.concept_distinctiveness,
    ]
    result.overall = sum(scores) / len(scores)
    result.is_passing = result.overall >= threshold and min(scores) >= 4.0
    return result

In [ ]:
results = evaluate_image(
    4.5,
    brand_guidelines_image,
    idea_image_path
)

In [ ]:
results

# Tool 5: revise_image_prompt

In [ ]:
class PromptRevision(BaseModel):
    revised_prompt: str = Field(
        description="Standalone image-generation prompt for the next attempt."
    )
    changes: List[str] = Field(
        description="Concrete changes made compared with the previous prompt."
    )
    preserved_constraints: List[str] = Field(
        description="Important brand/user constraints intentionally preserved."
    )
    retry_focus: str = Field(
        description="Short summary of what the next generation should improve most."
    )

In [ ]:
def revise_image_prompt(
    previous_prompt: str,
    evaluation_result: EvaluationResult,
    brand_analysis: BrandAnalysis,
    user_description: str,
    brand_guidelines_image: str | None = None,
    failed_image: str | None = None,
) -> PromptRevision:
    revision_prompt = f"""
    You are a senior prompt engineer for UI image generation.

    Revise the previous image-generation prompt so the next mockup fixes the
    evaluation failures while preserving everything that already matches the
    brand and strengthening the concept's uniqueness.

    User request:
    {user_description}

    Brand analysis JSON:
    {brand_analysis.model_dump_json()}

    Previous image-generation prompt:
    {previous_prompt}

    Evaluation JSON:
    {evaluation_result.model_dump_json()}

    Revision rules:
    - Return a complete standalone prompt, not a patch or commentary.
    - Preserve the brand identity, palette, typography, component language,
      content specificity, and any strengths reflected in the evaluation.
    - Preserve the intended concept direction unless the evaluation says that
      direction is the problem.
    - Directly address low-scoring criteria and every actionable item in
      prompt_improvements.
    - Replace generic or placeholder content with brand-specific names, labels,
      CTAs, metrics, section names, and domain language from the brand analysis
      when available.
    - Do not solve feedback by copying the reference screenshots more closely.
      If distinctiveness is weak, change composition, hierarchy, section order,
      or user journey while keeping the brand tokens stable.
    - Do not introduce new unrelated sections, stock portraits, browser chrome,
      lorem ipsum, placeholder names, watermarks, or visual styles absent from
      the reference.
    - Keep the prompt concise enough for image generation, but specific enough
      to constrain copy, layout, contrast, spacing, and component consistency.
    - Include a brief negative instruction section inside revised_prompt.
    """

    content = [{"type": "input_text", "text": revision_prompt}]

    if brand_guidelines_image:
        brand_guidelines_image_b64 = encode_image(brand_guidelines_image)
        content.append(
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{brand_guidelines_image_b64}",
            }
        )

    if failed_image:
        failed_image_b64 = encode_image(failed_image)
        content.append(
            {
                "type": "input_image",
                "image_url": f"data:image/png;base64,{failed_image_b64}",
            }
        )

    response = client.responses.parse(
        model=MODEL_NAME,
        input=[{"role": "user", "content": content}],
        text_format=PromptRevision,
    )

    return response.output_parsed

# Run the functions in a loop

In [ ]:
class AttemptRecord(BaseModel):
    concept_index: int
    concept_title: str
    attempt: int
    image_path: str
    prompt: str
    evaluation: EvaluationResult
    revision: PromptRevision | None = None


class DesignRunResult(BaseModel):
    run_dir: str
    final_image_path: str
    selected_concept_index: int
    selected_concept_title: str
    passed: bool
    best_attempt: AttemptRecord
    attempts: List[AttemptRecord]
    ideas: Ideas
    brand_analysis: BrandAnalysis

In [ ]:
def finalize_design_run(
    run_dir: str,
    brand_analysis: BrandAnalysis,
    ideas: Ideas,
    selected_concept_index: int,
    attempts: List[AttemptRecord],
    best_attempt: AttemptRecord,
    passed: bool,
) -> DesignRunResult:
    run_path = Path(run_dir)
    run_path.mkdir(parents=True, exist_ok=True)

    final_image_path = run_path / "final_image.png"
    final_image_path.write_bytes(Path(best_attempt.image_path).read_bytes())

    (run_path / "brand_analysis.json").write_text(
        brand_analysis.model_dump_json(indent=2), encoding="utf-8"
    )
    (run_path / "concepts.json").write_text(
        ideas.model_dump_json(indent=2), encoding="utf-8"
    )
    (run_path / "attempts.jsonl").write_text(
        "\n".join(attempt.model_dump_json() for attempt in attempts),
        encoding="utf-8",
    )
    (run_path / "final_prompt.txt").write_text(best_attempt.prompt, encoding="utf-8")
    (run_path / "final_evaluation.json").write_text(
        best_attempt.evaluation.model_dump_json(indent=2), encoding="utf-8"
    )

    selected_concept = ideas.ideas[selected_concept_index]
    report = f"""# Design Run Summary

Passed: {passed}
Selected concept: {selected_concept.title}
Differentiation angle: {selected_concept.differentiation_angle}
Best attempt: {best_attempt.attempt}
Best score: {best_attempt.evaluation.overall}
Final image: {final_image_path}

Feedback:
{best_attempt.evaluation.feedback}
"""
    (run_path / "summary.md").write_text(report, encoding="utf-8")

    return DesignRunResult(
        run_dir=str(run_path),
        final_image_path=str(final_image_path),
        selected_concept_index=selected_concept_index,
        selected_concept_title=selected_concept.title,
        passed=passed,
        best_attempt=best_attempt,
        attempts=attempts,
        ideas=ideas,
        brand_analysis=brand_analysis,
    )


def run_design_generation(
    user_description: str,
    brand_guidelines_image: str = "output/brand_image.jpg",
    run_dir: str = "output/design_run",
    threshold: float = 4.6,
    max_attempts: int = 3,
    num_concepts: int = 3,
    concept_index: int = 0,
) -> DesignRunResult:
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")
    if num_concepts < 2:
        raise ValueError("num_concepts should be at least 2 for idea exploration")

    run_path = Path(run_dir)
    run_path.mkdir(parents=True, exist_ok=True)

    brand_analysis = brand_analysis_tool(brand_guidelines_image)
    ideas = generate_design_concepts(
        user_description=user_description,
        brand_guidelines_image=brand_guidelines_image,
        brand_analysis=brand_analysis,
        num_concepts=num_concepts,
    )

    print(f"Generated {len(ideas.ideas)} concept(s).")

    if not ideas.ideas:
        raise RuntimeError("No design concepts were generated")
    if concept_index < 0 or concept_index >= len(ideas.ideas):
        raise IndexError(
            f"concept_index must be between 0 and {len(ideas.ideas) - 1}"
        )

    selected_concept = ideas.ideas[concept_index]
    print(f"Running concept {concept_index + 1}: {selected_concept.title}")
    print(f"Differentiation: {selected_concept.differentiation_angle}")
    current_prompt = f"""{selected_concept.prompt}

Negative instructions:
{selected_concept.negative_prompt}
"""

    attempts: List[AttemptRecord] = []
    best_attempt: AttemptRecord | None = None

    for attempt_number in range(1, max_attempts + 1):
        print(f"Attempt {attempt_number}/{max_attempts}: generating image...")
        image_path = run_path / f"concept_{concept_index + 1}_attempt_{attempt_number}.png"
        generated_image_path = generate_idea_image(
            current_prompt,
            brand_guidelines_image,
            output_path=str(image_path),
        )

        if generated_image_path is None:
            raise RuntimeError(f"Image generation failed on attempt {attempt_number}")

        evaluation = evaluate_image(
            threshold,
            brand_guidelines_image,
            generated_image_path,
        )

        print(
            f"Attempt {attempt_number}: overall={evaluation.overall:.2f}, "
            f"passing={evaluation.is_passing}"
        )

        attempt_record = AttemptRecord(
            concept_index=concept_index,
            concept_title=selected_concept.title,
            attempt=attempt_number,
            image_path=generated_image_path,
            prompt=current_prompt,
            evaluation=evaluation,
        )
        attempts.append(attempt_record)

        if best_attempt is None or evaluation.overall > best_attempt.evaluation.overall:
            best_attempt = attempt_record

        if evaluation.is_passing:
            return finalize_design_run(
                run_dir=str(run_path),
                brand_analysis=brand_analysis,
                ideas=ideas,
                selected_concept_index=concept_index,
                attempts=attempts,
                best_attempt=attempt_record,
                passed=True,
            )

        if attempt_number < max_attempts:
            revision = revise_image_prompt(
                previous_prompt=current_prompt,
                evaluation_result=evaluation,
                brand_analysis=brand_analysis,
                user_description=user_description,
                brand_guidelines_image=brand_guidelines_image,
                failed_image=generated_image_path,
            )
            print(f"Revision focus: {revision.retry_focus}")
            attempt_record.revision = revision
            current_prompt = revision.revised_prompt

    if best_attempt is None:
        raise RuntimeError("No attempts were completed")

    return finalize_design_run(
        run_dir=str(run_path),
        brand_analysis=brand_analysis,
        ideas=ideas,
        selected_concept_index=concept_index,
        attempts=attempts,
        best_attempt=best_attempt,
        passed=False,
    )

In [ ]:
user_description = "Create a UI landing page for this cooking app"
brand_guidelines_image = "output/guide.png"

In [ ]:
run_result = run_design_generation(user_description, max_attempts=3, num_concepts=2)